# F2 — Figures for the experimental candidate panel (F1)

**Question.** How do the F1 candidates look against their own uncertainty, against the whole atlas,
and on the SH3 fold, so the panel can be shown (thesis, supervisor, collaborators) before any bench
work is planned?

**What it does.** Draws three figures from files that already exist. Nothing is recomputed, and no
number from A0–F1 changes.
1. **figF1_candidate_pairs:** each destabilising pick next to its same-domain control, with the 90 %
   conformal interval and the 1 kcal/mol threshold.
2. **figF2_atlas_context:** where the picks and controls sit in the distribution of lower bounds of all
   6,884 atlas variants.
3. **figF3_position_map:** the picks on the SH3 family landscape from B6 (mean predicted damage per
   aligned position), showing how concentrated the panel is along the fold.

**Inputs:** `F1/experimental_candidates.csv`, `B3/atlas_predictions.parquet`, `B6/family_position_map.csv`.
**Outputs:** `figF1…figF3` (PNG + PDF), `candidate_positions.csv`, `RESULTS_F2.md`.
**Runtime:** seconds, CPU. Explanation: `docs/F2_candidate_figures.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'F2', 'Figures for the experimental candidate panel', resume=RESUME)
print(C.env_info())

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110,
                     'savefig.dpi': 300, 'savefig.bbox': 'tight'})
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = '#2f6fb5', '#d8792a', '#3f8f5e', '#c0392b', '#7a5aa8', '#9a9a9a'
def save(fig, name):
    for ext in ('png', 'pdf'): fig.savefig(R.out(f'{name}.{ext}'))
    plt.show(); plt.close(fig)

CAND = pd.read_csv(R.res('F1', 'experimental_candidates.csv'))
A = pd.read_parquet(R.res('B3', 'atlas_predictions.parquet'))
FAM = pd.read_csv(R.res('B6', 'family_position_map.csv'))
CAND = CAND.merge(A[['variant_id', 'ref_pos', 'wt_aa', 'mut_aa']], on='variant_id', how='left')
CAND['label'] = CAND.gene + '  ' + CAND.wt_aa + CAND.pos.astype(int).astype(str) + CAND.mut_aa
CAND['upper'] = CAND.pred_ddG_fold + CAND.fold_halfwidth
PICK = CAND[CAND.mechanism.str.startswith('destabilising')].copy()
CTRL = CAND[CAND.mechanism.str.startswith('tolerated')].copy()
assert len(PICK) and len(CTRL), 'Run F1 first: no candidates found.'
print(f'{len(PICK)} picks, {len(CTRL)} controls, {len(A):,} atlas variants, {len(FAM)} aligned positions')

## 1. Each pick next to its same-domain control

In [ ]:
order = PICK.sort_values('fold_lo').gene.tolist()
fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(order) + 1.6))
for i, g in enumerate(order):
    for df, col, dy, mk in [(PICK, RED, 0.17, 'o'), (CTRL, BLUE, -0.17, 's')]:
        r = df[df.gene == g]
        if len(r) == 0: continue
        r = r.iloc[0]
        ax.hlines(i + dy, r.fold_lo, r.upper, color=col, lw=2.5, alpha=.55)
        ax.plot(r.pred_ddG_fold, i + dy, mk, color=col, ms=7)
        ax.text(r.upper + 0.12, i + dy, r.label.split('  ')[1], va='center', fontsize=8, color=col)
ax.axvline(1.0, color='k', ls='--', lw=1); ax.axvline(0, color=GREY, lw=.8)
ax.text(1.03, len(order) - 0.4, 'call threshold: lower bound ≥ 1 kcal/mol', fontsize=8)
ax.set_yticks(range(len(order)), order); ax.set_xlabel('predicted ΔΔG$_{fold}$ (kcal/mol), 90 % conformal interval')
ax.plot([], [], 'o', color=RED, label='destabilising pick'); ax.plot([], [], 's', color=BLUE, label='tolerated control, same domain')
ax.legend(frameon=False, loc='lower right', fontsize=8)
ax.set_title(f'{len(PICK)} bench picks, each paired with a neutral control from the same SH3 domain', loc='left', weight='bold')
save(fig, 'figF1_candidate_pairs')

## 2. The panel against the whole atlas

In [ ]:
PCT = {v: float((A.fold_lo < l).mean() * 100) for v, l in zip(PICK.variant_id, PICK.fold_lo)}
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bins = np.linspace(A.fold_lo.min(), A.fold_lo.max(), 70)
ax.hist(A.fold_lo, bins=bins, color=GREY, alpha=.6, label=f'all {len(A):,} atlas variants')
ax.hist(A.fold_lo[A.fold_lo >= 1], bins=bins, color=ORANGE, alpha=.7, label=f'called destabilising ({int((A.fold_lo >= 1).sum())})')
top = ax.get_ylim()[1]
for x in PICK.fold_lo: ax.plot([x, x], [0, top * .12], color=RED, lw=2)
for x in CTRL.fold_lo: ax.plot([x, x], [0, top * .12], color=BLUE, lw=2)
ax.plot([], [], color=RED, lw=2, label='picks'); ax.plot([], [], color=BLUE, lw=2, label='controls')
ax.axvline(1.0, color='k', ls='--', lw=1); ax.set_yscale('log')
ax.set_xlabel('lower 90 % bound of predicted ΔΔG$_{fold}$ (kcal/mol)'); ax.set_ylabel('variants (log scale)')
ax.set_title(f'Picks are the extreme tail: every pick above the {min(PCT.values()):.1f}th percentile of the atlas',
             loc='left', weight='bold')
ax.legend(frameon=False, fontsize=8); save(fig, 'figF2_atlas_context')

## 3. Where the picks sit on the SH3 fold (B6 family landscape)

In [ ]:
POS = PICK.dropna(subset=['ref_pos']).groupby('ref_pos').agg(picks=('variant_id', 'size'),
                                                             genes=('gene', lambda s: ', '.join(s))).reset_index()
POS.to_csv(R.out('candidate_positions.csv'), index=False)
fig, ax = plt.subplots(figsize=(10, 4.2))
cols = np.where(FAM.buried, ORANGE, GREY)
ax.bar(FAM.ref_pos, FAM.mean_pred, color=cols, width=.8)
ax.bar(FAM.ref_pos[FAM.interface], FAM.mean_pred[FAM.interface], color='none', edgecolor=PURPLE, lw=1.5, width=.8)
ymax = FAM.mean_pred.max()
for _, r in POS.iterrows():
    ax.annotate(f'{int(r.picks)}', (r.ref_pos, FAM.set_index('ref_pos').mean_pred.get(r.ref_pos, ymax) + .08),
                ha='center', color=RED, weight='bold')
    ax.plot(r.ref_pos, -0.12, '^', color=RED, ms=9, clip_on=False)
# proxy handles must be real artists: ax.bar([], []) yields a container with no patch,
# so the legend falls back to a default swatch and the colours are wrong.
HANDLES = [Patch(facecolor=ORANGE, label='buried'),
           Patch(facecolor=GREY, label='exposed'),
           Patch(facecolor='none', edgecolor=PURPLE, lw=1.5, label='peptide interface'),
           Line2D([], [], marker='^', color=RED, ls='none', ms=9, label='F1 picks (number above bar)')]
ax.set_xlabel('aligned position (GRB2-SH3 frame)'); ax.set_ylabel('mean predicted ΔΔG$_{fold}$, all substitutions')
ax.set_ylim(0, ymax * 1.2); ax.legend(handles=HANDLES, frameon=False, fontsize=8, ncol=4, loc='upper right')
TOPPOS = POS.sort_values('picks', ascending=False).iloc[0]
ax.set_title(f'{int(TOPPOS.picks)} of {len(PICK)} picks sit at aligned position {int(TOPPOS.ref_pos)}: '
             f'the panel covers {len(POS)} of {len(FAM)} positions', loc='left', weight='bold')
save(fig, 'figF3_position_map')
print(POS.to_string(index=False))

## Results

In [ ]:
for k, v in [('picks', len(PICK)), ('controls', len(CTRL)), ('positions_covered', len(POS)),
             ('aligned_positions', len(FAM)), ('max_picks_at_one_position', int(TOPPOS.picks)),
             ('most_used_position', int(TOPPOS.ref_pos)), ('min_pick_percentile', min(PCT.values())),
             ('picks_buried', int(PICK.ref_pos.map(FAM.set_index('ref_pos').buried).fillna(False).sum())),
             ('picks_to_proline', int((PICK.mut_aa == 'P').sum()))]:
    R.metric(k, v)
R.section('Figures', '', pd.DataFrame([('figF1_candidate_pairs', 'each pick vs its same-domain control, 90 % intervals'),
                                       ('figF2_atlas_context', 'picks and controls in the distribution of all atlas lower bounds'),
                                       ('figF3_position_map', 'picks on the B6 family landscape (aligned positions)')],
                                      columns=['file (PNG + PDF)', 'shows']))
R.section('Aligned positions used by the picks', 'Concentration check: one position dominating means the panel '
          'tests one structural mechanism.', POS)
R.finish([f"{len(PICK)} picks and {len(CTRL)} same-domain controls drawn; every pick is above the "
          f"{min(PCT.values()):.1f}th percentile of atlas lower bounds.",
          f"The picks cover {len(POS)} of {len(FAM)} aligned positions; {int(TOPPOS.picks)} sit at position {int(TOPPOS.ref_pos)}.",
          f"{int((PICK.mut_aa == 'P').sum())} of {len(PICK)} picks are substitutions to proline.",
          'Figures only: nothing upstream was re-run or changed.'])